# ViBioMIR - Kaggle pipeline by source ID range

Enable Internet and T4 x2. Add a Kaggle Secret named `HF_TOKEN` with read access to the team dataset, then Run All. The notebook contains its own pinned pipeline code. Its artifacts and `submission.zip` go under the range-specific `OUT` in the configuration cell.

The only input selection controls are `ID_START` and `ID_END`. Both bounds are inclusive. The HF reader keeps every unique Vietnamese record within the interval whose ID and URL match the official dataset. It scans every matching corpus shard, including shards with an unknown filename pattern. The width of the ID interval is **not** the number of valid documents; check `source_manifest.json` for the actual count. Choose a smaller interval for a short pilot.

Stages: pinned HF snapshot -> QC/prepare -> source-offset chunks -> BM25 and Qwen3-Embedding-8B -> RRF -> one Qwen3-Reranker-8B pass -> validated submission ZIP. There is no query translation, feedback loop or fine-tuning in this run. The two 8B models use NF4 on separate T4 cards at different stages. The embedding index uses 2048 MRL dimensions to reduce disk and search cost.

Run All resumes existing stages in the same session. To resume after Save Version, attach the prior output as a Kaggle Input and set `RESUME_FROM` to its run directory. Keep the same ID bounds and other configuration. Changing the interval changes `OUT` automatically; changing model or chunk settings for the same interval requires a new `OUT`. The dense stage checkpoints every 1024 chunks, and reranker scores are cached by query.

### Logs and resource usage

Each session writes `OUT/telemetry/<session-id>/events.jsonl`, `resources.jsonl`, `summary.json`, and one stdout/stderr `.log` per stage. Samples every `RESOURCE_INTERVAL_S` seconds include CPU, RAM, output and scratch disk, the stage process tree, and both GPUs (used/total VRAM, utilization, temperature, power). The summary includes stage status, elapsed time and resource peaks. Logs are written during execution and remain available if a stage fails or Kaggle stops. GPU metrics refer to the whole card.


In [ ]:
# ===== CẤU HÌNH DUY NHẤT CẦN SỬA =====
HF_REPO = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_REVISION = 'main'             # resolved to a fixed commit on first run
ID_START = 1_500_000             # inclusive source ID
ID_END = 1_999_999               # inclusive source ID; edit both bounds
SEED = 13                       # deterministic order if duplicate IDs occur
OUT = f'/kaggle/working/vibiomir_8b_ids_{ID_START}_{ID_END}'
RESUME_FROM = None              # e.g. '/kaggle/input/previous-output/vibiomir_pipeline'
ENCODER = 'qwen3-embedding-8b'   # NF4, MRL 2048 dimensions; use a new OUT if changed
RERANKER = 'qwen3-reranker-8b'  # one NF4 reranker
CHUNK_TARGET = 320
CHUNK_OVERLAP = 64
RERANK_K = 24                    # 1,200 * 24 = at most 28.8k expensive pairs
KEEP = 24
RESOURCE_INTERVAL_S = 20         # interval between resource snapshots

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('HF_TOKEN')
if not token:
    raise RuntimeError('Missing Kaggle Secret HF_TOKEN')
os.environ['HF_TOKEN'] = token
os.environ['HF_HOME'] = '/kaggle/temp/hf_home'  # model weights stay out of saved notebook Output
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONIOENCODING'] = 'utf-8'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.51,<5', 'accelerate>=1.6', 'huggingface_hub>=0.30',
                'pyarrow>=19', 'sentencepiece>=0.2', 'protobuf>=4.25',
                'bitsandbytes>=0.46,<0.51', 'psutil>=5.9'], check=True)
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError('Select Kaggle accelerator GPU T4 x2 before Run All.')
print('torch', torch.__version__, 'GPU', [torch.cuda.get_device_name(i) for i in range(2)])

In [ ]:
# Kaggle-only embedded pipeline code, including interval-aware HF snapshot.
# Rebuilding from the local generator would overwrite these Kaggle-only changes.
import base64, hashlib, io, shutil, zipfile
from pathlib import Path
CODE_SHA256 = 'e880222609f44d8775bf05ebcf5c4ceb16378a2a751f06b5878f9b40e939953e'
CODE_ZIP = (
    '...'
    'cAEAAAA='
)
payload = base64.b64decode(CODE_ZIP)
if hashlib.sha256(payload).hexdigest() != CODE_SHA256:
    raise ValueError('Code nhúng bị hỏng; build lại notebook.')
CODE_DIR = Path('/kaggle/working/vibiomir_code')
shutil.rmtree(CODE_DIR, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(payload)).extractall(CODE_DIR)
print('Code:', CODE_SHA256[:12], sum(1 for _ in CODE_DIR.rglob('*.py')), 'file .py')


In [ ]:
import atexit, hashlib, json, subprocess, threading, time, uuid
from datetime import datetime, timezone
from pathlib import Path
import psutil


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def sha256_file(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


OUT = Path(OUT)
if RESUME_FROM and not OUT.exists():
    origin = Path(RESUME_FROM)
    if not origin.is_dir():
        raise FileNotFoundError(origin)
    shutil.copytree(origin, OUT)
OUT.mkdir(parents=True, exist_ok=True)
CODE = CODE_DIR / 'scripts' / 'vibiomir.py'
if not CODE.is_file():
    raise FileNotFoundError(CODE)
if ENCODER != 'qwen3-embedding-8b' or RERANKER != 'qwen3-reranker-8b':
    raise ValueError('This Kaggle profile requires the pinned 8B encoder and reranker.')
if not 1 <= KEEP <= RERANK_K:
    raise ValueError('KEEP must be between 1 and RERANK_K.')
if not isinstance(ID_START, int) or not isinstance(ID_END, int) or not 0 <= ID_START <= ID_END:
    raise ValueError('Require integer IDs with 0 <= ID_START <= ID_END.')
if RESOURCE_INTERVAL_S < 5:
    raise ValueError('RESOURCE_INTERVAL_S must be at least 5 seconds.')

state_dir = OUT / 'stages'
state_dir.mkdir(exist_ok=True)
session_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
telemetry_dir = OUT / 'telemetry' / session_id
telemetry_dir.mkdir(parents=True, exist_ok=False)


class ResourceMonitor:
    """Sample system use while stages run; never reads or writes HF credentials."""
    def __init__(self, directory, interval):
        self.directory = directory
        self.interval = interval
        self.events_path = directory / 'events.jsonl'
        self.resources_path = directory / 'resources.jsonl'
        self.summary_path = directory / 'summary.json'
        self.started_utc = utc_now()
        self.started = time.monotonic()
        self.stage = 'startup'
        self.pid = None
        self.process_cache = {}
        self.stats = {}
        self.lock = threading.Lock()
        self.stop_event = threading.Event()
        self.closed = False
        self.event('session_start', interval_s=interval, output=str(OUT), code_sha256=CODE_SHA256)
        self.sample()
        self.worker = threading.Thread(target=self._loop, name='resource-monitor', daemon=True)
        self.worker.start()

    def event(self, kind, **details):
        record = {'utc': utc_now(), 'elapsed_s': round(time.monotonic()-self.started, 2),
                  'stage': self.stage, 'event': kind, **details}
        with self.lock:
            with self.events_path.open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')
        print(f'[telemetry] {record["stage"]}: {kind}', flush=True)

    def set_stage(self, stage, pid=None):
        with self.lock:
            self.stage = stage or 'idle'
            self.pid = pid
            self.process_cache = {}

    @staticmethod
    def _disk(path):
        if not Path(path).exists():
            return None
        usage = psutil.disk_usage(str(path))
        return {'used_gib': round(usage.used/2**30, 3),
                'total_gib': round(usage.total/2**30, 3),
                'free_gib': round(usage.free/2**30, 3)}

    @staticmethod
    def _gpu():
        query = ('index,name,utilization.gpu,memory.used,memory.total,'
                 'temperature.gpu,power.draw,power.limit')
        try:
            result = subprocess.run(
                ['nvidia-smi', f'--query-gpu={query}', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=8, check=True)
        except (FileNotFoundError, subprocess.SubprocessError) as error:
            return {'error': str(error)}
        devices = []
        for line in result.stdout.splitlines():
            fields = [item.strip() for item in line.split(',')]
            if len(fields) != 8:
                continue
            def number(value):
                try:
                    return float(value)
                except ValueError:
                    return None
            devices.append({'index': int(fields[0]), 'name': fields[1],
                            'util_percent': number(fields[2]),
                            'memory_used_mib': number(fields[3]),
                            'memory_total_mib': number(fields[4]),
                            'temperature_c': number(fields[5]),
                            'power_w': number(fields[6]),
                            'power_limit_w': number(fields[7])})
        return devices

    def sample(self):
        with self.lock:
            stage, pid = self.stage, self.pid
        memory = psutil.virtual_memory()
        processes = {'pid': pid, 'rss_gib': 0.0, 'cpu_core_percent': 0.0, 'count': 0}
        if pid is not None:
            try:
                root = psutil.Process(pid)
                for item in [root, *root.children(recursive=True)]:
                    try:
                        process = self.process_cache.setdefault(item.pid, item)
                        processes['rss_gib'] += process.memory_info().rss/2**30
                        processes['cpu_core_percent'] += process.cpu_percent(interval=None)
                        processes['count'] += 1
                    except (psutil.NoSuchProcess, psutil.AccessDenied):
                        pass
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
        processes['rss_gib'] = round(processes['rss_gib'], 3)
        processes['cpu_core_percent'] = round(processes['cpu_core_percent'], 1)
        record = {'utc': utc_now(), 'elapsed_s': round(time.monotonic()-self.started, 2),
                  'stage': stage, 'system': {
                      'cpu_percent': psutil.cpu_percent(interval=None),
                      'cpu_logical': psutil.cpu_count(logical=True),
                      'ram_used_gib': round(memory.used/2**30, 3),
                      'ram_total_gib': round(memory.total/2**30, 3),
                      'ram_percent': memory.percent,
                      'output_disk': self._disk(OUT),
                      'scratch_disk': self._disk('/kaggle/temp')},
                  'process_tree': processes, 'gpus': self._gpu()}
        with self.lock:
            with self.resources_path.open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(record, ensure_ascii=False) + '\n')
            stats = self.stats.setdefault(stage, self._blank_stats())
            stats['samples'] += 1
            count = stats['samples']
            stats['mean_system_cpu_percent'] = round(
                (stats['mean_system_cpu_percent']*(count-1) + record['system']['cpu_percent'])/count, 2)
            stats['peak_ram_gib'] = max(stats['peak_ram_gib'], record['system']['ram_used_gib'])
            stats['peak_process_rss_gib'] = max(stats['peak_process_rss_gib'], processes['rss_gib'])
            for gpu in record['gpus'] if isinstance(record['gpus'], list) else []:
                key = str(gpu['index'])
                for field, value in (('peak_gpu_memory_mib', gpu['memory_used_mib']),
                                     ('peak_gpu_util_percent', gpu['util_percent'])):
                    if value is not None:
                        stats[field][key] = max(stats[field].get(key, 0), value)
                if gpu['util_percent'] is not None:
                    count = stats['gpu_util_samples'].get(key, 0)
                    old = stats['mean_gpu_util_percent'].get(key, 0.0)
                    stats['gpu_util_samples'][key] = count + 1
                    stats['mean_gpu_util_percent'][key] = round(
                        (old*count + gpu['util_percent'])/(count+1), 2)
        return record

    def _loop(self):
        while not self.stop_event.wait(self.interval):
            try:
                self.sample()
            except Exception as error:
                self.event('monitor_error', error=repr(error))

    @staticmethod
    def _blank_stats():
        return {'samples': 0, 'peak_ram_gib': 0, 'peak_process_rss_gib': 0,
                'peak_gpu_memory_mib': {}, 'peak_gpu_util_percent': {},
                'mean_gpu_util_percent': {}, 'gpu_util_samples': {},
                'mean_system_cpu_percent': 0.0}

    def mark_stage(self, name, status, seconds, error=None):
        with self.lock:
            item = self.stats.setdefault(name, self._blank_stats())
            item.update(status=status, seconds=round(seconds, 2))
            if error is not None:
                item['error'] = str(error)

    def write_summary(self):
        with self.lock:
            summary = {'session': session_id, 'started_utc': self.started_utc,
                       'updated_utc': utc_now(), 'elapsed_s': round(time.monotonic()-self.started, 2),
                       'stages': self.stats, 'events': str(self.events_path),
                       'resources': str(self.resources_path)}
        self.summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    def close(self):
        if self.closed:
            return
        self.closed = True
        self.stop_event.set()
        self.worker.join(timeout=3)
        try:
            self.sample()
        except Exception:
            pass
        self.event('session_end')
        self.write_summary()


monitor = ResourceMonitor(telemetry_dir, RESOURCE_INTERVAL_S)
atexit.register(monitor.close)
monitor.event('run_config', repo=HF_REPO, id_start=ID_START, id_end=ID_END,
              encoder=ENCODER,
              reranker=RERANKER, chunk_target=CHUNK_TARGET,
              chunk_overlap=CHUNK_OVERLAP, rerank_k=RERANK_K, keep=KEEP,
              translation=False, feedback_rounds=0)
print('Telemetry:', telemetry_dir)

from huggingface_hub import HfApi
model_revisions_path = OUT / 'model_revisions.json'
monitor.set_stage('model_pin')
try:
    if model_revisions_path.exists():
        model_revisions = json.loads(model_revisions_path.read_text())
    else:
        api = HfApi(token=os.environ['HF_TOKEN'])
        model_revisions = {
            ENCODER: api.model_info('Qwen/Qwen3-Embedding-8B').sha,
            RERANKER: api.model_info('Qwen/Qwen3-Reranker-8B').sha,
        }
        model_revisions_path.write_text(json.dumps(model_revisions, indent=2))
    monitor.event('model_pin_complete', revisions={k: v[:12] for k, v in model_revisions.items()})
except BaseException as error:
    monitor.event('model_pin_failed', error_type=type(error).__name__, error=str(error))
    monitor.close()
    raise
finally:
    monitor.set_stage(None)
print('Pinned models:', {k: v[:12] for k, v in model_revisions.items()})


def run_stage(name, arguments, outputs, gpu=None, key_extra=None):
    command = [sys.executable, '-u', str(CODE), *map(str, arguments)]
    record = state_dir / (name + '.json')
    key = hashlib.sha256(json.dumps({'code': CODE_SHA256, 'cmd': command,
                                     'extra': key_extra}, sort_keys=True).encode()).hexdigest()
    monitor.set_stage(name)
    monitor.sample()
    monitor.event('stage_start', gpu=gpu, command=command, outputs=[str(p) for p in outputs])
    stage_started = time.monotonic()
    log_path = telemetry_dir / f'{name}.log'
    tail = []
    try:
        if record.exists():
            previous = json.loads(record.read_text())
            if previous['key'] != key:
                raise ValueError(f'{name}: configuration/code changed; use a new OUT.')
            if all(Path(p).is_file() and sha256_file(p) == digest
                   for p, digest in previous['outputs'].items()):
                elapsed = time.monotonic()-stage_started
                monitor.mark_stage(name, 'reused', elapsed)
                monitor.event('stage_reused', validation_s=round(elapsed, 2))
                print(f'[{name}] verified; reuse', flush=True)
                return
        env = os.environ.copy()
        if gpu is not None:
            env['CUDA_VISIBLE_DEVICES'] = str(gpu)
        print(f'[{name}]', ' '.join(map(str, arguments)), flush=True)
        with log_path.open('a', encoding='utf-8') as log:
            log.write(f'\n===== {utc_now()} START {name} =====\n')
            log.flush()
            process = subprocess.Popen(command, cwd=CODE_DIR, env=env,
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                       text=True, encoding='utf-8', errors='replace', bufsize=1)
            monitor.set_stage(name, process.pid)
            monitor.event('process_started', pid=process.pid, log=str(log_path))
            monitor.sample()
            try:
                for line in process.stdout:
                    tail.append(line.rstrip())
                    if len(tail) > 20:
                        tail.pop(0)
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = process.wait()
            except BaseException:
                process.terminate()
                process.wait()
                raise
            finally:
                process.stdout.close()
            log.write(f'===== {utc_now()} EXIT {returncode} =====\n')
        if returncode:
            raise RuntimeError(f'{name} exited with code {returncode}; inspect {log_path}')
        missing = [str(p) for p in outputs if not Path(p).is_file()]
        if missing:
            raise RuntimeError(f'{name} completed without required output: {missing}')
        hashes = {str(p): sha256_file(p) for p in outputs}
        record.write_text(json.dumps({'key': key, 'outputs': hashes,
                                      'seconds': time.monotonic()-stage_started}, indent=2))
        monitor.mark_stage(name, 'completed', time.monotonic()-stage_started)
        monitor.event('stage_completed', seconds=round(time.monotonic()-stage_started, 2),
                      output_bytes={str(p): Path(p).stat().st_size for p in outputs})
    except BaseException as error:
        monitor.mark_stage(name, 'failed', time.monotonic()-stage_started, error)
        monitor.event('stage_failed', error_type=type(error).__name__, error=str(error),
                      log_tail=tail,
                      seconds=round(time.monotonic()-stage_started, 2), log=str(log_path))
        monitor.close()
        raise
    finally:
        if not monitor.closed:
            monitor.sample()
            monitor.set_stage(None)
            monitor.write_summary()

## 1. Pin an HF corpus snapshot and official ID/URL mapping

Read `corpus/*.parquet` from one HF commit using ranged reads. Keep only records inside the inclusive source-ID interval, with a matching official ID/URL pair and Vietnamese content. Clearly disjoint `ids_<min>_<max>` shards are skipped; shards with other names are scanned. Save the selected records to local Parquet without downloading all HF shards or the `archives/`, `diagnostics/`, and `teach/` trees.


In [ ]:
source = OUT / 'source'
run_stage('hf_snapshot', ['hf-snapshot', '--repo', HF_REPO, '--revision', HF_REVISION,
                          '--id-start', ID_START, '--id-end', ID_END,
                          '--all-in-range', '--seed', SEED, '--sample-mode', 'bounded',
                          '--out', source],
          [source/'source_manifest.json', source/'input_vi.parquet',
           source/'links_corpus_vi_en_zh.parquet', source/'query.parquet'])
print('Source:', json.loads((source/'source_manifest.json').read_text())['counts'])


## 2. Chuẩn hóa, QC, chia chunk

`prepare` loại trang lỗi/template và chốt `corpus.parquet`; `chunk` giữ `char_start`, `char_end`, `chunk_order` để bài nộp cắt lại chính xác từ văn bản nguồn.


In [ ]:
corpus_dir = OUT / 'corpus'
corpus = corpus_dir / 'corpus.parquet'
chunks_dir = OUT / 'chunks'
chunks = chunks_dir / 'chunks.parquet'
run_stage('prepare', ['prepare', '--source', source/'input_vi.parquet', '--out', corpus_dir], [corpus, corpus_dir/'prepare_manifest.json'],
          key_extra=json.loads((source/'source_manifest.json').read_text())['input_sha256'])
run_stage('chunk', ['chunk', '--corpus', corpus, '--out', chunks_dir,
                    '--target-tokens', CHUNK_TARGET, '--overlap-tokens', CHUNK_OVERLAP,
                    '--max-tokens', 768],
          [chunks, chunks_dir/'chunk_manifest.json'],
          key_extra=hashlib.file_digest(open(corpus, 'rb'), 'sha256').hexdigest())
print('Documents:', json.loads((corpus_dir/'prepare_manifest.json').read_text())['documents_kept'])
print('Chunks:', json.loads((chunks_dir/'chunk_manifest.json').read_text())['chunks'])


## 3. Chỉ mục BM25 và dense

BM25 chạy CPU. Qwen3-Embedding-8B NF4 mã hóa chunk trên GPU 0, ghi checkpoint mỗi 1024 chunk. Giai đoạn này có thể mất nhiều giờ; chạy lại cùng Output dùng checkpoint đã ghi.


In [ ]:
index = OUT / 'index'
chunk_hash = hashlib.file_digest(open(chunks, 'rb'), 'sha256').hexdigest()
run_stage('bm25', ['index', 'bm25', '--chunks', chunks, '--out', index],
          [index/'bm25.npz', index/'bm25_manifest.json'], key_extra=chunk_hash)
run_stage('dense', ['index', 'dense', '--chunks', chunks, '--out', index,
                    '--encoder', ENCODER, '--device', 'cuda', '--quantize', '4bit',
                    '--max-length', 512, '--batch-size', 2, '--shard', 1024, '--resume',
                    '--revision', model_revisions[ENCODER]],
          [index/'embeddings.npy', index/'dense_manifest.json'], gpu=0,
          key_extra=chunk_hash)


## 4. BM25 + dense → RRF → một lần rerank

Qwen3-Reranker-8B NF4 trên GPU 1 chỉ chấm tối đa `RERANK_K` chunk cho mỗi query. Cache `scores.sqlite` giữ điểm query đã xong. Không có vòng feedback/rewrite ở lần đầu.


In [ ]:
retrieval = OUT / 'retrieval'
queries = source / 'query.parquet'
run_stage('retrieve', ['retrieve', '--chunks', chunks, '--index', index,
                       '--queries', queries, '--out', retrieval,
                       '--reranker', RERANKER, '--reranker-revision', model_revisions[RERANKER],
                       '--reranker-quantize', '4bit', '--device', 'cuda',
                       '--search-device', 'cuda', '--keyword-variant',
                       '--top-k', 200, '--fuse-k', 250, '--rerank-k', RERANK_K,
                       '--keep', KEEP, '--candidate-per-doc', 3,
                       '--rerank-max-length', 512, '--batch-size', 2,
                       '--rerank-batch', 1, '--search-block', 8192,
                       '--query-block', 32],
          [retrieval/'candidates.jsonl', retrieval/'retrieval_manifest.json'], gpu=1,
          key_extra={'chunks': chunk_hash,
                     'bm25': hashlib.file_digest(open(index/'bm25.npz','rb'), 'sha256').hexdigest(),
                     'dense': hashlib.file_digest(open(index/'dense_manifest.json','rb'), 'sha256').hexdigest(),
                     'queries': hashlib.file_digest(open(queries,'rb'), 'sha256').hexdigest()})


## 5. Validate and create the submission ZIP

Creates one `submission/submission.zip` using the recall profile. The notebook does not submit it automatically.


In [ ]:
links = source/'links_corpus_vi_en_zh.parquet'
target = OUT / 'submission'
run_stage('submit_recall',
          ['submit', '--candidates', retrieval/'candidates.jsonl',
           '--chunks', chunks, '--corpus', corpus, '--queries', queries,
           '--links', links, '--profile', 'recall', '--require-complete',
           '--out', target],
          [target/'submission.zip', target/'submission.json',
           target/'submission_manifest.json'],
          key_extra=hashlib.file_digest(open(retrieval/'candidates.jsonl','rb'), 'sha256').hexdigest())
summary = json.loads((target/'submission_manifest.json').read_text())
if summary['queries_without_candidates'] or summary['validation']['dropped_docs'] or summary['validation']['dropped_chunks']:
    monitor.event('submission_validation_failed', validation=summary['validation'])
    monitor.close()
    raise RuntimeError('Incomplete or invalid submission; inspect manifest')
monitor.event('submission_ready', zip=str(target/'submission.zip'), submission=summary['submission'])
monitor.close()
print('ready:', target/'submission.zip', summary['submission'])
print('Telemetry:', telemetry_dir)
print('Stage resource summary:', json.dumps(json.loads(monitor.summary_path.read_text())['stages'], ensure_ascii=False, indent=2))
from IPython.display import FileLink, display

display(FileLink(str(target/'submission.zip')))
display(FileLink(str(monitor.summary_path)))
display(FileLink(str(monitor.resources_path)))